In [10]:
from data_frame.optimization.partition.partition_optimizer import PartitionOptimizer
import time
from data_frame.spark_utils import get_spark
from pyspark.sql import functions as F

In [11]:
spark = get_spark(app_name="Spark partitioning and bucketing")

In [16]:

# Create DataFrame
df = spark.range(0, 1000000) \
    .withColumn("key", F.col("id") % 100) \
    .withColumn("value", F.rand())

df_other = spark.range(0, 100000) \
    .withColumn("key", F.col("id") % 100) \
    .withColumn("other_value", F.rand())

## 1. Repartitioning for Performance

In [18]:
print(f"Initial partitions: {df.rdd.getNumPartitions()}")

# Repartition for better distribution
df_repartitioned = PartitionOptimizer.repartition_for_optimization(
    df, num_partitions=10, partition_cols=["key"]
)
print(f"After repartition: {df_repartitioned.rdd.getNumPartitions()}")

Initial partitions: 8
After repartition: 10


In [19]:
# Join without optimization
start = time.time()
result1 = df.join(df_other, "key").count()
time1 = time.time() - start

In [20]:
# Join with optimized partitioning
df_other_optimized = PartitionOptimizer.repartition_for_optimization(
    df_other, num_partitions=10, partition_cols=["key"]
)
start = time.time()
result2 = df_repartitioned.join(df_other_optimized, "key").count()
time2 = time.time() - start

print(f"Join without optimization: {time1:.2f}s")
print(f"Join with optimization: {time2:.2f}s")
print(f"Improvement: {time1/time2:.1f}x")

Join without optimization: 1.66s
Join with optimization: 1.89s
Improvement: 0.9x


## 2. Coalescing Partitions

In [14]:
# Coalesce for fewer partitions (e.g., before writing to file)
df_coalesced = PartitionOptimizer.coalesce_partitions(df, 5)
print(f"After coalesce (5 partitions): {df_coalesced.rdd.getNumPartitions()}")

# Optimize partition size
df_optimized = PartitionOptimizer.optimize_partition_size(df, target_size_mb=64)
print(f"Optimal partitions based on size: {df_optimized.rdd.getNumPartitions()}")

After coalesce (5 partitions): 5
Optimal partitions based on size: 1


## 3. Bucketing Example

In [15]:
# Bucketing is typically done when writing
df.write \
    .mode("overwrite") \
    .bucketBy(10, "key") \
    .sortBy("value") \
    .saveAsTable("bucketed_table")

# Read bucketed table
df_bucketed = spark.table("bucketed_table")
print(f"Bucketed table partitions: {df_bucketed.rdd.getNumPartitions()}")
df_bucketed.printSchema()

26/03/18 12:55:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/18 12:55:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Bucketed table partitions: 8
root
 |-- id: long (nullable = true)
 |-- key: long (nullable = true)
 |-- value: double (nullable = true)

